### Преобразование вектора состояния в орбитальные элементы

In [1]:
from astroquery.jplhorizons import Horizons

from astropy.coordinates import SkyCoord
from astropy import units as u

from astropy.time import Time

from datetime import datetime

from math import *
import numpy as np
import pandas as pd

# rebound - это Вам уже известно - модуль, 
# позволяющий использовать целый набор интеграторов для типичных астрономических приложений. 
import rebound

# https://github.com/matthewholman/assist
# Удобный пакет для доступа к динамическим эфемеридам JPL DE441.
# Применяется для численного интегрирования движения "безмассовых" частиц (астероидов, метеороидов).
# То есть никакого численного интегрирования положений Солнца и планет не производится. 
# Это только интерполяция на основе таблиц JPL DE441. 
import assist

# подключаем эфемериду (файл для Солнца, Луны, планет и файл для крупных астероидов) 
ephem = assist.Ephem("/usr/local/etc/de/linux_p1550p2650.440", "/usr/local/etc/de/sb441-n16.bsp")

# мы пока ограничимся планетами, чтобы добавить их векторы состояния;
# для этого создадим список; 
ss_bodies = [
        "Sun",
        "Mercury",
        "Venus",
        "Earth",
        "Moon",
        "Mars",
        "Jupiter",
        "Saturn",
        "Uranus",
        "Neptune",
        "Pluto"]

In [2]:
# текущее всемирное время по часам компьютера
ut = Time(datetime.utcnow(), scale='utc')
print(ut)

# извлекаем юлианский день в шкале UTC
ep = ut.jd
# юлианский день в системе барицентрического времени TDB
ep_tdb = ut.tdb.jd
# разность барицентрического времени и всемирного в секундах (это для прикола, но в каждом приколе - доля прикола)
(ep_tdb-ep)*86400

2023-09-22 07:58:38.944807


69.18240487575531

In [3]:
# идентификатор астероида
ast = '2014 HK129'
# формируем объект класса Horizons для добычи вектора состояния. 
# Horizons - это система эфемерид NASA JPL https://ssd.jpl.nasa.gov/horizons/app.html#/
# то есть этот объект организует web-запросы к Horizons без браузера и специальных знаний о get- и post- запросах 
# тут в запрос надо в качестве аргумента передать TDB согласно описанию (https://astroquery.readthedocs.io/en/latest/api/astroquery.jplhorizons.HorizonsClass.html#astroquery.jplhorizons.HorizonsClass), location='@0' означает, 
# что вектор состояния будет вычислен относительно барицентра Солнечной системы
obj = Horizons(id=ast, id_type='smallbody', location='@0', epochs=ep_tdb)
# вектор состояния (опорная плоскость - средний экватор на эпоху J2000)
pos_ast = obj.vectors(refplane='earth', delta_T=True)

# иногда удобнее превратить табличку в numpy-массив
ast_state = np.array([pos_ast['x'][0], pos_ast['y'][0],pos_ast['z'][0],pos_ast['vx'][0],\
                      pos_ast['vy'][0],pos_ast['vz'][0]])
# показать таблицу, полученную в результате запроса
pos_ast

targetname,datetime_jd,datetime_str,H,G,delta_T,x,y,z,vx,vy,vz,lighttime,range,range_rate
---,d,---,mag,---,s,AU,AU,AU,AU / d,AU / d,AU / d,d,AU,AU / d
str12,float64,str30,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
(2014 HK129),2460209.833195917,A.D. 2023-Sep-22 07:59:48.1272,21.1,0.15,69.1824,0.6516584778154412,-1.916509564318318,-0.896987438805858,0.009534147456891079,-0.0005432117388538765,-0.001445673763122113,0.01278759514130558,2.214103463528256,0.00386198290141993


In [4]:
# создаем REBOUND-симуляцию
sim = rebound.Simulation()
# конвертируем вектор состояния в набор переменных просто для понимания (тут можно было не плодить сущностей:))
x,y,z,vx,vy,vz = ast_state
# добавляем в нашу симуляцию вектор состояния (массу не указываем, значит это безмассовая точка)
sim.add(x=x,y=y,z=z,vx=vx,vy=vy,vz=vz)
# устанавливаем текущий момент времени для нашей симуляции
# ephem.jd_ref - начальный момент времени для DE441
print(ephem.jd_ref)
# поэтому нужно установить время так, чтобы оно совпало с текущим для использования эфемерид NASA JPL
sim.t = ep_tdb - ephem.jd_ref

# создаем симуляцию для Солнца, Луны и больших планет
sim_ss = rebound.Simulation()
# в цикле добавляем в симуляцию векторы состояния всех этих тел Солнечной системы (Солнца, Луны и больших планет)
for ss_body in ss_bodies:
#     print(ss_body)
    sim_ss.add(ephem.get_particle(ss_body, ep_tdb - ephem.jd_ref))

# добавляем наш астероид 
sim_ss.add(sim.particles[0])

# ex = assist.Extras(sim, ephem)
# ex.integrate_or_interpolate(ep_tdb - ephem.jd_ref)

# вычисляем орбитальные элементы для нашего вектора состояния с помощью функции, представленной в REBOUND
o = sim_ss.particles[11].calculate_orbit(primary=sim_ss.particles[0])

# представляем элементы в виде массива
el=np.array([o.a, o.e, o.P, degrees(o.inc), degrees(o.omega), degrees(o.Omega), 
             degrees(o.f),  o.T+ephem.jd_ref, ep_tdb])

2451545.0


In [5]:
# организуем табличку и помним, что параметры ориентации орбиты будут взяты относительно плоскости экватора 
# (а не эклиптики - как это принято в традиционных приложениях)
orb_df = pd.DataFrame(columns = ['AST','SRC','a(a.u.)','e','P(day)','inc(deg)','omega(deg)','Omega(deg)','true_anomaly(deg)','ep_per(day)','epoch(day)'], index=range(1))
orb_df.iloc[0][0]=ast;
orb_df.iloc[0][1]='JPL';
orb_df.iloc[0][2:]= el
orb_df

,AST,SRC,a(a.u.),e,P(day),inc(deg),omega(deg),Omega(deg),true_anomaly(deg),ep_per(day),epoch(day)
0,2014 HK129,JPL,1.700155,0.488754,809.712915,23.892199,123.835346,16.728219,148.263319,2451306.015381,2460209.833196


In [6]:
# собственно надо попробовать преобразовать эти элементы в набор элементов относительно плоскости эклиптики.
# Потом можно сравнить с элементами в базе данных https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2014%20HK129
# только тут надо брать гелиоцентрический вектор состояния...

In [7]:
# создаем REBOUND-симуляцию
sim = rebound.Simulation()
# конвертируем вектор состояния в набор переменных просто для понимания (тут можно было не плодить сущностей:))
x,y,z,vx,vy,vz = ast_state

c = SkyCoord(x = x, y = y, z = z, representation_type='cartesian', frame='icrs')
cv = SkyCoord(x = vx, y = vy, z = vz, representation_type='cartesian', frame='icrs')
c_ecl = c.transform_to('barycentricmeanecliptic')
cv_ecl = cv.transform_to('barycentricmeanecliptic')

# добавляем в нашу симуляцию вектор состояния (массу не указываем, значит это безмассовая точка)
sim.add(x =c_ecl.cartesian.x, y = c_ecl.cartesian.y, z = c_ecl.cartesian.z,
          vx =cv_ecl.cartesian.x, vy = cv_ecl.cartesian.y, vz = cv_ecl.cartesian.z)
# устанавливаем текущий момент времени для нашей симуляции
# ephem.jd_ref - начальный момент времени для DE441
print(ephem.jd_ref)
# поэтому нужно установить время так, чтобы оно совпало с текущим для использования эфемерид NASA JPL
sim.t = ep_tdb - ephem.jd_ref

# создаем симуляцию для Солнца, Луны и больших планет
sim_ss = rebound.Simulation()
# в цикле добавляем в симуляцию векторы состояния всех этих тел Солнечной системы (Солнца, Луны и больших планет)
for ss_body in ss_bodies:
#     print(ss_body)
    ss_p = ephem.get_particle(ss_body, ep_tdb - ephem.jd_ref)
    c = SkyCoord(x = ss_p.x, y = ss_p.y, z = ss_p.z, representation_type='cartesian', frame='icrs')
    cv = SkyCoord(x = ss_p.vx, y = ss_p.vy, z = ss_p.vz, representation_type='cartesian', frame='icrs')
    c_ecl = c.transform_to('barycentricmeanecliptic')
    cv_ecl = cv.transform_to('barycentricmeanecliptic')
    sim_ss.add(m = ss_p.m, x =c_ecl.cartesian.x, y = c_ecl.cartesian.y, z = c_ecl.cartesian.z,
                  vx =cv_ecl.cartesian.x, vy = cv_ecl.cartesian.y, vz = cv_ecl.cartesian.z)
    #

# добавляем наш астероид 
sim_ss.add(sim.particles[0])


# вычисляем орбитальные элементы для нашего вектора состояния с помощью функции, представленной в REBOUND
o = sim_ss.particles[11].calculate_orbit(primary=sim_ss.particles[0])

# представляем элементы в виде массива
el=np.array([o.a, o.e, o.P, degrees(o.inc), degrees(o.omega), degrees(o.Omega), 
             degrees(o.f),  o.T+ephem.jd_ref, ep_tdb])

2451545.0


In [8]:
# организуем табличку и помним, что параметры ориентации орбиты будут взяты относительно плоскости экватора 
# (а не эклиптики - как это принято в традиционных приложениях)
orb_df = pd.DataFrame(columns = ['AST','SRC','a(a.u.)','e','P(day)','inc(deg)','omega(deg)','Omega(deg)','true_anomaly(deg)','ep_per(day)','epoch(day)'], index=range(1))
orb_df.iloc[0][0]=ast;
orb_df.iloc[0][1]='JPL';
orb_df.iloc[0][2:]= el
orb_df

,AST,SRC,a(a.u.),e,P(day),inc(deg),omega(deg),Omega(deg),true_anomaly(deg),ep_per(day),epoch(day)
0,2014 HK129,JPL,1.700155,0.488754,809.712915,6.709714,45.336333,93.838052,148.263319,2451306.015381,2460209.833196
